# Mode Selection
Set finetuning = 1 if you want to finetune, or set finetuning = 0 if you want to just load the saved model to get inference.

In [ ]:
finetuning = 0
load_and_run = not finetuning

# Drive Mount

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Load Data

In [ ]:
import pandas as pd
data = pd.read_csv('/kaggle/input/gb-all-necessary/train_test_senti_data.csv')
data

# Train, Test, Validation Split
We are going to consider the data as test data if the prediction between original sentence and NER converted sentence become unequal.

In [ ]:
temp = data[data['test']!= 1]
test = data[data['test']== 1]

In [ ]:
validation = temp.groupby('_original_label', group_keys=False).apply(lambda x: x.sample(frac=0.2, random_state=101))
train = temp.drop(validation.index)

In [ ]:
train

In [ ]:
validation

In [ ]:
test

# Minimizing Joint Loss

## Load Pretrained Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transform
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig
import pandas as pd
import numpy as np
import transformers
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn.functional as F
from torchsummary import summary
from tqdm import tqdm
from tqdm import tqdm
import torch
import torch.optim as optim
import torch.nn.functional as F
from collections import OrderedDict
from datetime import datetime
from transformers import get_linear_schedule_with_warmup, AdamW

def load_transformer_based_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path ,output_hidden_states=True)
    return tokenizer, model
model_path = "ka05ar/banglabert-sentiment"
tokenizer, b_model = load_transformer_based_model(model_path)
print('Total Pretrained tokens: ',len(tokenizer.get_vocab()))
tokenizer.add_tokens(['<Name>' ,'<Gender>'])
b_model.resize_token_embeddings(len(tokenizer))
print('Total number of tokens after adding the new ones: ',len(tokenizer.get_vocab()))

## Creating Custom Data Loader

In [ ]:
class BertDataset(Dataset):
    def __init__(self, data, tokenizer,max_length = 512):
        super(BertDataset, self).__init__()

        self.train_csv= data
        self.tokenizer=tokenizer
        self.target=self.train_csv.iloc[:,2]
        self.max_length=max_length

    def __len__(self):
        return len(self.train_csv)

    def __getitem__(self, index):

        text1 = self.train_csv.iloc[index,0] #0th column male
        text2 = self.train_csv.iloc[index,1] #1st column female

        inputs = self.tokenizer.batch_encode_plus(
            [text1, text2] ,
            pad_to_max_length=True,
            add_special_tokens=True,
            return_attention_mask=True,
            max_length=self.max_length,
            truncation = True
        )
        ids = inputs["input_ids"]
        token_type_ids = inputs["token_type_ids"]
        masks = inputs["attention_mask"]

        return torch.tensor(ids[0], dtype=torch.long), torch.tensor(token_type_ids[0], dtype=torch.long), torch.tensor(masks[0], dtype=torch.long), torch.tensor(ids[1], dtype=torch.long), torch.tensor(token_type_ids[1], dtype=torch.long), torch.tensor(masks[1], dtype=torch.long), torch.tensor(self.train_csv.iloc[index, 2], dtype=torch.long)

BATCH_SIZE = 4

dataset_train= BertDataset(train, tokenizer)
dataset_val= BertDataset( validation, tokenizer)
dataset_test= BertDataset(test, tokenizer)
train_loader=DataLoader(dataset=dataset_train,batch_size=BATCH_SIZE, shuffle=True)
validation_loader=DataLoader(dataset=dataset_val,batch_size=BATCH_SIZE, shuffle=False)
test_loader=DataLoader(dataset=dataset_test,batch_size=BATCH_SIZE, shuffle=False)

## Creating Custom Model

In [ ]:
class BERT(nn.Module):
    def __init__(self, model):
        super(BERT, self).__init__()
        self.bert_model = model
        self.dropout = nn.Dropout(0.2)
        self.out = nn.Linear(768, 1)

    def forward(self,ids1, mask1, token_type_ids1, ids2, mask2, token_type_ids2, mode):
        if mode == 'training':
          ids = torch.vstack((ids1, ids2))
          mask = torch.vstack((mask1, mask2))
          token_type_ids = torch.vstack((token_type_ids1, token_type_ids2))
          o2= self.bert_model(ids,mask,token_type_ids).hidden_states[0]
          o2= o2.max(dim=1).values
          original_embedding, ner_embedding = torch.vsplit(o2, 2)
          # print(original_embedding.shape, ner_embedding.shape)
          o2 = self.dropout(original_embedding)
          out= self.out(o2)
          # print(o2.shape)

          return out, original_embedding, ner_embedding

        if mode == 'inference':
          o2= self.bert_model(ids1, mask1, token_type_ids1).hidden_states[0]
          oe= o2.max(dim=1).values
          o2 = self.dropout(oe)
          out= self.out(o2)
          return out, oe

masked_model=BERT(b_model)

In [ ]:
# #debug
# i1, t1,m1,i2,t2,m2,l = next(iter(train_loader))
# print(i1.shape)
# pred, oe, nere = masked_model(i1, t1,m1,i2,t2,m2, 'training')
# print(pred.shape, oe.shape, nere.shape)
# pred, oe = masked_model(i1, t1,m1,None,None,None, 'inference')
# print(pred.shape, oe.shape)

## Creating Utility Class for Tracking Training Stat

In [ ]:
from collections import OrderedDict
from collections import namedtuple
from itertools import product
import json
from torch.utils.tensorboard import SummaryWriter
import time
import torchvision
import pandas as pd
import torch
from IPython.display import display
from IPython.display import clear_output

class RunBuilder():
    @staticmethod
    def get_runs(params):
        Run = namedtuple('Run', params.keys())


        runs = []
        for v in product(*params.values()):
            runs.append(Run(*v))


        return runs


class RunManager():
    def __init__(self):
        self.epoch_count = 0
        self.epoch_ce_loss = 0
        self.epoch_sim_loss = 0
        self.epoch_loss = 0
        self.epoch_num_correct = 0
        self.epoch_start_time = None

        self.epoch_valid_ce_loss = 0.0
        self.epoch_valid_sim_loss = 0.0
        self.epoch_valid_loss = 0.0
        self.epoch_num_valid_correct = 0

        self.run_params = None
        self.run_count = 0
        self.run_data = []
        self.run_start_time = None

        self.network = None
        self.loader = None
        self.validation_loader = None
        self.tb = None

    def begin_run(self, run, network, loader, validation_loader):
        self.run_start_time = time.time()

        self.run_params = run
        self.run_count += 1

        self.network = network
        self.loader = loader
        self.validation_loader = validation_loader
        self.tb = SummaryWriter(comment=f'-{run}')



    def end_run(self):
        self.tb.close()
        self.epoch_count = 0

    def begin_epoch(self):
        self.epoch_start_time = time.time()
        self.epoch_count += 1
        self.epoch_loss = 0
        self.epoch_ce_loss = 0
        self.epoch_sim_loss = 0
        self.epoch_num_correct = 0
        self.epoch_valid_ce_loss = 0
        self.epoch_valid_sim_loss = 0
        self.epoch_valid_loss = 0
        self.epoch_num_valid_correct = 0

    def end_epoch(self):
        epoch_duration = time.time() - self.epoch_start_time
        run_duration = time.time() - self.run_start_time

        ce_loss = self.epoch_ce_loss/len(self.loader.dataset)
        sim_loss = self.epoch_sim_loss/len(self.loader.dataset)
        loss = self.epoch_loss/len(self.loader.dataset)
        accuracy = self.epoch_num_correct/len(self.loader.dataset)
        val_ce_loss = self.epoch_valid_ce_loss/len(self.validation_loader.dataset)
        val_sim_loss = self.epoch_valid_sim_loss/len(self.validation_loader.dataset)
        val_loss = self.epoch_valid_loss/len(self.validation_loader.dataset)
        val_accuracy = self.epoch_num_valid_correct/len(self.validation_loader.dataset)

        self.tb.add_scalar('CE Loss', ce_loss, self.epoch_count)
        self.tb.add_scalar('Sim Loss', sim_loss, self.epoch_count)
        self.tb.add_scalar('Loss', loss, self.epoch_count)
        self.tb.add_scalar('Accuracy', accuracy, self.epoch_count)
        self.tb.add_scalar('Validation CE Loss', val_ce_loss, self.epoch_count)
        self.tb.add_scalar('Validation Sim Loss', val_sim_loss, self.epoch_count)
        self.tb.add_scalar('Validation Loss', val_loss, self.epoch_count)
        self.tb.add_scalar('Validation Accuracy', val_accuracy, self.epoch_count)

        for name, param in self.network.named_parameters():
            self.tb.add_histogram(name, param, self.epoch_count)

        results = OrderedDict()
        results["run"] = self.run_count
        results["epoch"] = self.epoch_count
        results["ce_loss"] = ce_loss
        results["sim_loss"] = sim_loss
        results["loss"] = loss
        results["accuracy"] = accuracy
        results["validation ce loss"] = val_ce_loss
        results["validation sim loss"] = val_sim_loss
        results["validation loss"] = val_loss
        results["validation accuracy"] = val_accuracy
        results["epoch duration"] = epoch_duration
        results["run duration"] = run_duration
        for k,v in self.run_params._asdict().items(): results[k] = v
        self.run_data.append(results)
        df = pd.DataFrame.from_dict(self.run_data, orient='columns')

        clear_output(wait=True)
        display(df)

    def track_loss(self, ce_loss, sim_loss, loss):
        self.epoch_ce_loss += ce_loss.item()#*self.loader.batch_size
        self.epoch_sim_loss += sim_loss.item()#*self.loader.batch_size
        self.epoch_loss += loss.item()#*self.loader.batch_size

    def track_num_correct(self, preds, labels):
        self.epoch_num_correct += self._get_num_correct(preds, labels)

    def track_validation_loss(self, ce_loss, sim_loss, loss):
        self.epoch_valid_ce_loss += ce_loss.item()#*self.loader.batch_size
        self.epoch_valid_sim_loss += sim_loss.item()#*self.loader.batch_size
        self.epoch_valid_loss += loss.item()#*self.loader.batch_size

    def track_num_validation_correct(self, preds, labels):
        self.epoch_num_valid_correct += self._get_num_correct(preds, labels)

    @torch.no_grad()
    def _get_num_correct(self, preds, labels):
        return torch.where(preds >= 0, 1, 0).eq(labels).sum().item()

    def save(self, fileName):
        pd.DataFrame.from_dict(
            self.run_data,
            orient='columns').to_csv(rf'{fileName}.csv')

## Creating Training Loop

In [ ]:
SAVE_MODEL = True
LAMBDA = 1
def my_trainer(train_loader, validation_loader, network, LR = 1e-5 ,epochs = 5, BATCH_SIZE = 16,  fresh_training = True, validation = True):
    params = OrderedDict(
        lr = [LR],
        batch_size =[BATCH_SIZE],
        device = ['cuda' if torch.cuda.is_available() else 'cpu'],
    )

    m = RunManager()
    for run in RunBuilder.get_runs(params):
        device = torch.device(run.device)
        network = network.to(device)
        # optimizer = optim.Adam(network.parameters(), lr=run.lr)

        loss_fn = nn.BCEWithLogitsLoss()
        loss_fn_2 = nn.CosineSimilarity()

        total_steps = len(train_loader) * epochs
        optimizer = optim.Adam(network.parameters(), lr= run.lr)

        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = 0, num_training_steps = total_steps)
        if not fresh_training:
            load_model(network, optimizer, PATH = '')
        m.begin_run(run, network, train_loader, validation_loader)
        for epoch in range(epochs):
            network.train(True)
            print(f'Epoch {epoch + 1} : ', end = '\t')
            m.begin_epoch()
            for batch in tqdm(train_loader):
                input_ids1, token_type_ids1, attention_masks1, input_ids2, token_type_ids2, attention_masks2, labels = batch
                input_ids1 = input_ids1.to(device)
                token_type_ids1 = token_type_ids1.to(device)
                attention_masks1 = attention_masks1.to(device)

                input_ids2 = input_ids2.to(device)
                token_type_ids2 = token_type_ids2.to(device)
                attention_masks2 = attention_masks2.to(device)

                labels = labels.type(torch.LongTensor)
                labels = labels.to(device)

                preds,original_hidden_states, ner_hidden_states = network(input_ids1, attention_masks1, token_type_ids1, input_ids2, attention_masks2, token_type_ids2, 'training')
                labels = labels.type_as(preds).view(-1, 1)
                # print(labels.shape)
                # print('----------')
                # print(preds.shape)


                loss1 = loss_fn(preds, labels)
                loss2 = 1 - torch.mean(torch.abs(loss_fn_2(original_hidden_states,ner_hidden_states)))
                loss = loss1 + (LAMBDA*loss2)
                # print(preds.shape, labels.shape)
                # print(preds, labels)


                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
                scheduler.step()


                m.track_loss(loss1, loss2, loss)
                m.track_num_correct(preds, labels)


            network.eval()
            for batch in tqdm(validation_loader):
                input_ids1, token_type_ids1, attention_masks1, input_ids2, token_type_ids2, attention_masks2, labels = batch
                input_ids1 = input_ids1.to(device)
                token_type_ids1 = token_type_ids1.to(device)
                attention_masks1 = attention_masks1.to(device)
                input_ids2 = input_ids2.to(device)
                token_type_ids2 = token_type_ids2.to(device)
                attention_masks2 = attention_masks2.to(device)

                labels = labels.type(torch.LongTensor)
                labels = labels.to(device)
                with torch.no_grad():
                    preds,original_hidden_states, ner_hidden_states = network(input_ids1, attention_masks1, token_type_ids1, input_ids2, attention_masks2, token_type_ids2, 'training')
                    labels = labels.type_as(preds).view(-1, 1)
                    loss1 = loss_fn(preds, labels)
                    loss2 = 1 - torch.mean(torch.abs(loss_fn_2(original_hidden_states,ner_hidden_states)))
                    loss = loss1 + (LAMBDA*loss2)
                    m.track_validation_loss(loss1, loss2, loss)
                    m.track_num_validation_correct(preds, labels)
                # break #not doing validation here




            m.end_epoch()
            network.train(False)
        m.end_run()
    if SAVE_MODEL == True:
        save_model(network, optimizer)

    s = datetime.now().strftime("%d-%m-%Y %I:%M:%p").replace(" ", "_").replace(":", "_")
    PATH  = rf'results_{s}'
    m.save(PATH)
    return


def save_model(network, optimizer):
    s = datetime.now().strftime("%d-%m-%Y %I:%M:%p").replace(" ", "_").replace(":", "_")
    PATH  = rf'Senti_JLO.pth'
    torch.save({
    'model_state_dict': network.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    }, PATH)
    print(f'Saved model and optimizer at {PATH}')
    return

def load_model(network, optimizer, PATH ):
    checkpoint = torch.load(PATH)
    network.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(rf'Loaded model and optimizer from {PATH}')
    else:
        print(rf'Loaded model from {PATH}')

    return

def get_parameter_info(network, details = False):
    print('Total parameters:' ,(sum(p.numel() for p in network.parameters())))
    print('Total trainable parameters:' ,(sum(p.numel() for p in network.parameters() if p.requires_grad)))
    if details:
        print('---------------------------------\nDetailed Parameter Info: ')

        for name, parameter in network.named_parameters():
            print('name, parameter.requires_grad, parameter.is_cuda, parameter.size(): ', name, parameter.requires_grad, parameter.is_cuda, parameter.size())
            print('---------------------------------')
    return


if finetuning:
  my_trainer(train_loader, validation_loader, masked_model, 0.0001, epochs = 15, BATCH_SIZE = BATCH_SIZE, fresh_training = True)

## Load Saved Model

In [ ]:
#Load Model
from tqdm import tqdm
import torch
import torch.optim as optim
import torch.nn.functional as F
from collections import OrderedDict
from datetime import datetime
from transformers import get_linear_schedule_with_warmup, AdamW

def load_model(network, optimizer, PATH ):
    checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
    network.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(rf'Loaded model and optimizer from {PATH}')
    else:
        print(rf'Loaded model from {PATH}')

    return


if load_and_run:
  load_model(masked_model, None, PATH = '/kaggle/input/jlo_senti/pytorch/default/1/Senti_JLO.pth')

## Inference

In [ ]:
import warnings 
warnings.filterwarnings('ignore') 
def get_prediction(data_loader, network, choice = 'original'):
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  device = torch.device(device)
  network = network.to(device)
  network.eval()
  l = []
  emb = []
  for batch in tqdm(data_loader):
    input_ids1, token_type_ids1, attention_masks1, input_ids2, token_type_ids2, attention_masks2, labels = batch
    input_ids1 = input_ids1.to(device)
    token_type_ids1 = token_type_ids1.to(device)
    attention_masks1 = attention_masks1.to(device)
    input_ids2 = input_ids2.to(device)
    token_type_ids2 = token_type_ids2.to(device)
    attention_masks2 = attention_masks2.to(device)
    labels = labels.type(torch.LongTensor)
    labels = labels.to(device)
    with torch.no_grad():
        if choice == 'male':
          preds, hidden_states = network(input_ids1, attention_masks1,token_type_ids1, None, None, None, 'inference')
        else:
          preds, hidden_states = network(input_ids2, attention_masks2,token_type_ids2, None, None, None, 'inference')

        labels = labels.type_as(preds).view(-1, 1)
        preds =  torch.where(preds >= 0, 1, 0)
        l.append(preds)
        emb.append(hidden_states)
  return l, torch.vstack(emb).cpu().numpy()
dataset_test= BertDataset(test, tokenizer, max_length=200)
test_loader=DataLoader(dataset=dataset_test,batch_size=16, shuffle = False)

p, emb_org = get_prediction(test_loader, masked_model, 'male')
test['pred_male_fod'] = [x[0] for pred in p for x in pred.tolist()]
p, emb_ner = get_prediction(test_loader, masked_model, 'female')
test['pred_female_fod'] = [x[0] for pred in p for x in pred.tolist()]
x = sum(test['pred_male_fod']!=test['_original_label'])
print('For Male:\nTotal test data: ', len(test),'Total mismatch: ', x,'Accuracy: ', 1- (x/len(test))) 
x = sum(test['pred_female_fod']!=test['_original_label'])
print('For Female:\nTotal test data: ', len(test),'Total mismatch: ', x,'Accuracy: ', 1- (x/len(test)))
print('Male Female Mismatch: ', sum(test['pred_male_fod']!=test['pred_female_fod']))

## JLO Result

In [ ]:
import numpy as np
np.random.seed(0)

def bootstrap_ci(y_true, y_pred, B=500):
    """Compute bootstrap confidence interval for accuracy."""
    N = len(y_true)
    original_acc = np.mean(y_pred == y_true)

    bootstrap_accs = []
    
    for _ in range(B):
        indices = np.random.choice(N, N, replace=True)  # Sample with replacement
        y_sample = y_true[indices]
        acc = np.mean(y_pred[indices] == y_sample)
        bootstrap_accs.append(acc)
    # Compute 95% confidence interval
    lower, upper = np.percentile(bootstrap_accs, [2.5, 97.5])
    
    return original_acc, (lower, upper), np.mean(bootstrap_accs)

y_true = test['_original_label'].values  # Ground truth labels
y_pred = test['pred_male_fod'].values
acc, ci, ma = bootstrap_ci(y_true, y_pred)
print('Male: ')
print(f"Accuracy: {acc:.4f}")
print(f"95% Confidence Interval: {ci}")
print(f"mean accuracy: {ma}")

y_true = test['_original_label'].values
y_pred = test['pred_female_fod'].values
acc, ci, ma = bootstrap_ci(y_true, y_pred)

print('\nFemale: ')
print(f"Accuracy: {acc:.4f}")
print(f"95% Confidence Interval: {ci}")
print(f"mean accuracy: {ma}")

print('Male Female Mismatch: ', sum(test['pred_male_fod']!=test['pred_female_fod']))
test.to_csv('revision_jlo_senti_result.csv', index = False)
print('saved...')

In [ ]:
# Statistical Parity Difference (SPD) and Equal Opportunity Difference (EOD) Calculation
import numpy as np
import pandas as pd
np.random.seed(0)

test = pd.read_csv('revision_jlo_senti_result.csv')


# Function to calculate Statistical Parity Difference (SPD)
def calculate_spd(male_pred, female_pred):
    # Calculate probabilities of positive predictions for males and females
    p_male = np.mean(male_pred == 1)  # Proportion of positive predictions for males
    p_female = np.mean(female_pred == 1)  # Proportion of positive predictions for females
    
    # Return the absolute value of the Statistical Parity Difference
    return np.abs(p_male - p_female)

# Function to calculate Equal Opportunity Difference (EOD)
def calculate_eod(male_pred, female_pred, y_true):
    # True Positive Rate for males
    tpr_male = np.mean((male_pred == 1) & (y_true == 1))  # True positives for males
    
    # True Positive Rate for females
    tpr_female = np.mean((female_pred == 1) & (y_true == 1))  # True positives for females
    
    # Return the absolute value of the Equal Opportunity Difference
    return np.abs(tpr_male - tpr_female)

# Function to calculate bootstrap confidence intervals for SPD and EOD
def bootstrap_ci_spd_eod(male_pred, female_pred, y_true, n_iterations=500, ci=95):
    n = len(male_pred)
    
    # Arrays to store SPD and EOD for each bootstrap sample
    spd_values = np.zeros(n_iterations)
    eod_values = np.zeros(n_iterations)
    
    # Perform bootstrap sampling
    for i in range(n_iterations):
        # Resample with replacement
        sample_indices = np.random.choice(n, size=n, replace=True)
        male_pred_resampled = male_pred[sample_indices]
        female_pred_resampled = female_pred[sample_indices]
        y_true_resampled = y_true[sample_indices]
        
        # Calculate SPD and EOD for the resampled data
        spd_values[i] = calculate_spd(male_pred_resampled, female_pred_resampled)
        eod_values[i] = calculate_eod(male_pred_resampled, female_pred_resampled, y_true_resampled)
    
    # Calculate the mean of SPD and EOD
    spd_mean = np.mean(spd_values)
    eod_mean = np.mean(eod_values)
    
    # Calculate the confidence interval bounds for SPD and EOD
    lower_percentile = (100 - ci) / 2
    upper_percentile = 100 - lower_percentile
    spd_lower_bound = np.percentile(spd_values, lower_percentile)
    spd_upper_bound = np.percentile(spd_values, upper_percentile)
    eod_lower_bound = np.percentile(eod_values, lower_percentile)
    eod_upper_bound = np.percentile(eod_values, upper_percentile)
    
    return (np.abs(spd_mean), np.abs(spd_lower_bound), np.abs(spd_upper_bound)), (np.abs(eod_mean), np.abs(eod_lower_bound), np.abs(eod_upper_bound))


y_true = test._original_label.values
male_pred = test.pred_male_fod.values
female_pred = test.pred_female_fod.values

# Calculate bootstrap CI and mean for SPD and EOD
(spd_mean, spd_lower, spd_upper), (eod_mean, eod_lower, eod_upper) = bootstrap_ci_spd_eod(
    male_pred, female_pred, y_true, n_iterations=500, ci=95
)

print(f"SPD 95% Confidence Interval: [{spd_lower:.3f}, {spd_upper:.3f}]\tSPD Mean: {spd_mean:.3f}")
print('---------------------------------------------------------------------------------------------')
print(f"EOD 95% Confidence Interval: [{eod_lower:.3f}, {eod_upper:.3f}]\tEOD Mean: {eod_mean:.3f}")

## Sentence Embedding Visualization for Original Sentence and NER Converted Sentence

In [ ]:
#for visualizing training data embedding
dataset_train= BertDataset(train, tokenizer, max_length=200)
train_loader=DataLoader(dataset=dataset_train,batch_size=16, shuffle = False)
p, emb_ori = get_prediction(train_loader, masked_model, 'original')
p, emb_ner = get_prediction(train_loader, masked_model, 'ner')

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px

tsne_cls = TSNE(n_components=2, random_state=42).fit_transform(emb_ori)

labels = train['original_lebel']   # Extract labels
review_texts = train['Original Sentence']  # Extract review texts

# Prepare the data for Plotly plots
plot_data_cls = {'t-SNE Dimension 1': tsne_cls[:, 0],
                 't-SNE Dimension 2': tsne_cls[:, 1],
                 'Label': labels,
                 'Text': review_texts}

# Customize colorbar to show integer labels
colorbar_tickvals = list(range(1, 6))
colorbar_ticktext = [str(val) for val in colorbar_tickvals]

# Plotting with Plotly - [CLS] Token Embeddings
fig_cls = px.scatter(plot_data_cls, x='t-SNE Dimension 1', y='t-SNE Dimension 2',
                     color='Label', hover_data={'t-SNE Dimension 1': False, 't-SNE Dimension 2': False, 'Text': True})


fig_cls.update_layout(title='t-SNE of Embeddings Original Training Data with Labels', height=700,  font=dict(
        size=20,
))
fig_cls.update_traces(marker=dict(size=10, opacity=0.9))

fig_cls.update_layout(coloraxis_colorbar=dict(
    tickvals=colorbar_tickvals,
    ticktext=colorbar_ticktext
))

fig_cls.show()


In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px

tsne_cls = TSNE(n_components=2, random_state=42).fit_transform(emb_ner)

labels = train['original_lebel']   # Extract labels
review_texts = train['Converted Sentence']  # Extract review texts

# Prepare the data for Plotly plots
plot_data_cls = {'t-SNE Dimension 1': tsne_cls[:, 0],
                 't-SNE Dimension 2': tsne_cls[:, 1],
                 'Label': labels,
                 'Text': review_texts}

# Customize colorbar to show integer labels
colorbar_tickvals = list(range(1, 6))
colorbar_ticktext = [str(val) for val in colorbar_tickvals]

# Plotting with Plotly - [CLS] Token Embeddings
fig_cls = px.scatter(plot_data_cls, x='t-SNE Dimension 1', y='t-SNE Dimension 2',
                     color='Label', hover_data={'t-SNE Dimension 1': False, 't-SNE Dimension 2': False, 'Text': True})


fig_cls.update_layout(title='t-SNE of Embeddings NER Converted Training Data with Labels', height=700,  font=dict(
        size=20,
))
fig_cls.update_traces(marker=dict(size=10, opacity=0.9))

fig_cls.update_layout(coloraxis_colorbar=dict(
    tickvals=colorbar_tickvals,
    ticktext=colorbar_ticktext
))

fig_cls.show()


In [ ]:
test.to_csv('/content/drive/MyDrive/#Research/# GB/senti_reSult_a2.csv', index = False)